In [1]:
import numpy as np
import opt_einsum as oe
import matplotlib.pyplot as plt
import scipy as sp

from quant_rotor.CCC_integration_methods.Dense.de_solve_one_thermal import integration_scheme
import quant_rotor.CCC_integration_methods.Dense.thermofield_boltz_funcs as bz

from quant_rotor.Hamiltonian_models.Dense.operators import rotor_operators, heisenberg_operators
from quant_rotor.Hamiltonian_models.Dense.hamiltonian import hamiltonian_dense

from quant_rotor.Hamiltonian_models.Dense.basis_transform import combine_transform, energy_transform, TF_transform

from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_periodic import t_periodic
from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_periodic_fast import t_periodic as t_periodic_fast

from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_exact import t_1_amplitude_guess_ground_state, t_2_amplitude_guess_ground_state, amplitute_energy, intermediate_normalisation

from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_sub_class import (
    QuantumSimulation,
    SimulationParams,
    TensorData,
)
from quant_rotor.CCC_iterative_methods.Dense.t_amplitudes_sub_class_linked import (
    QuantumSimulation_linked,
    SimulationParams_linked,
    TensorData_linked,
)

from quant_rotor.CCC_integration_methods.Dense.stat_mech_thermo import U as U_stat

In [2]:
np.set_printoptions(precision=8)
np.set_printoptions(suppress=True)
np.set_printoptions(linewidth=np.inf)
np.set_printoptions(threshold=np.inf)

# CCC Iterative

## Initialization 

In [483]:
n_comb_sites = 2
site_comb = 2
site = n_comb_sites ** site_comb

state = 3
state_comb = state**n_comb_sites
state_TF = state ** 2

g = 0.1

periodic = False

In [484]:
K, V = rotor_operators(state, g)
H = hamiltonian_dense(site_comb, K, V, V, periodic)

eig_val, eig_vec = np.linalg.eigh(H)

In [485]:
# K_e, V_e, _ = energy_transform(K, V, V)
# K_e_TF, V_e_TF, _ = TF_transform(K, V, V)

In [486]:
# H_e_TF = hamiltonian_dense(site_comb, K_e_TF, V_e_TF, V_e_TF, periodic)

# eig_val_TF, eig_vec_TF = np.linalg.eigh(H_e_TF)

In [487]:
K_comb, V_comb_xy, V_comb_yx = combine_transform(2, K, V)
H_comb = hamiltonian_dense(site_comb, K_comb, V_comb_xy, V_comb_yx, periodic)

eig_val_comb, eig_vec_comb = np.linalg.eigh(H_comb)

In [488]:
K_e_comb, V_e_comb_xy, V_e_comb_yx = energy_transform(K_comb, V_comb_xy, V_comb_yx)

V_e_comb_xy = V_e_comb_xy.reshape(state_comb, state_comb, state_comb, state_comb)
V_e_comb_yx = V_e_comb_yx.reshape(state_comb, state_comb, state_comb, state_comb)

# for i in range(state_comb):
#     K_e_comb[i, i] = K_e_comb[i, i] - K_e_comb[0, 0]
#     for j in range(state_comb):
#         V_e_comb_xy[i, j, i, j] = V_e_comb_xy[i, j, i, j] - V_e_comb_xy[0, 0, 0, 0]
#         V_e_comb_yx[i, j, i, j] = V_e_comb_yx[i, j, i, j] - V_e_comb_yx[0, 0, 0, 0]

In [489]:
H_e_comb = hamiltonian_dense(site_comb, K_e_comb, V_e_comb_xy.reshape(state_comb**2, state_comb**2), V_e_comb_yx.reshape(state_comb**2, state_comb**2), periodic)

eig_val_e_comb, eig_vec_e_comb = np.linalg.eigh(H_e_comb)

In [490]:
print(eig_val)
# print(eig_val_TF)
print(eig_val_comb)
print(eig_val_e_comb)

[-0.00623059  0.9         0.95        1.05        1.1         2.          2.          2.          2.00623059]
[-0.01871957  0.826175    0.91012439  0.93223084  0.95943633  1.02128658  1.05515109  1.0717262   1.14896987  1.77668144  1.82484481  1.82484481  1.88676906  1.89936305  1.90780543  1.90780543  1.94752693  1.99236788  1.99484802  1.99484802  1.99563308  1.99704835  1.99891854  1.99891854  2.00621434  2.04748551  2.08725922  2.08725922  2.09938773  2.11023711  2.16759366  2.16759366  2.22254409  2.84896179  2.8591057   2.85929902  2.87192236  2.87928932  2.88695012  2.88726209  2.92192236  2.92200465  2.94257991  2.95685845  2.95695103  2.96731801  2.96800667  2.9709467   2.97928932  3.02071068  3.03149085  3.03669921  3.03700393  3.04414505  3.04418701  3.06568524  3.07807764  3.08343738  3.11555877  3.11576129  3.12071068  3.12807764  3.15097722  3.15113716  3.17257246  4.          4.          4.          4.          4.          4.00088883  4.00109572  4.00109572  4.00504716  

In [491]:
t_1_exact = t_1_amplitude_guess_ground_state(state_comb, site_comb, eig_vec_e_comb, eig_val_e_comb)
t_2_exact = t_2_amplitude_guess_ground_state(state_comb, site_comb, eig_vec_e_comb, eig_val_e_comb)

In [492]:
# T_1, T_2, R_1, R_2 = t_periodic(site_comb, state_comb, K_e_comb, V_e_comb_xy, V_e_comb_yx, "original", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

In [493]:
E = amplitute_energy(site_comb, state_comb, periodic, K_e_comb, V_e_comb_xy, V_e_comb_yx, np.copy(t_1_exact), np.copy(t_2_exact))

In [494]:
E - eig_val_e_comb[0]

np.complex128(0j)

## Initialization TF.

In [3]:
site = 2

state = 3
state_TF = state ** 2

g = 0.1

periodic = False

In [4]:
K, V = rotor_operators(state, g)
H = hamiltonian_dense(site, K, V, V, periodic)

eig_val, eig_vec = np.linalg.eigh(H)

In [5]:
def U_func(beta):
    U, U_vec, factor_n = bz.thermofield_change_of_basis_new(K, beta)

    # U, Q= bz.thermofield_change_of_basis(K)

    I = np.eye(state)

    K_prim = np.einsum('pq,mw->pmqw', K, I,  optimize='optimal').reshape(state_TF, state_TF)

    K_tilda = U.T @ K_prim @ U

    V_tensor = V.reshape(state, state, state, state)

    V_prim = np.einsum('pqrs,mw,nv->pmqnrwsv', V_tensor, I, I,  optimize='optimal').reshape(state_TF**2, state_TF**2)

    V_grouped = V_prim.reshape(state**2, state**2, state**2, state**2)

    V_tilda = np.einsum('Mi,Wj,ijab,aN,bV->MWNV', U.T, U.T, V_grouped, U, U, optimize='optimal').reshape(state_TF**2, state_TF**2)

    return U, K_tilda, V_tilda, factor_n

In [6]:
U, K_tilda, V_tilda, factor_n = U_func(0)


V_tilda = V_tilda.reshape(state_TF, state_TF, state_TF, state_TF)

for i in range(state_TF):
    K_tilda[i, i] = K_tilda[i, i] - K_tilda[0, 0]
    for j in range(state_TF):
        V_tilda[i, j, i, j] = V_tilda[i, j, i, j] - V_tilda[0, 0, 0, 0]

K_tilda = K_tilda
V_tilda = V_tilda.reshape(state_TF**2, state_TF**2)

H_TF = hamiltonian_dense(site, K_tilda, V_tilda, V_tilda, periodic)

eig_val_TF, eig_vec_TF = np.linalg.eigh(H_TF)

/Users/gilfrim/Desktop/Projects/Computational_Analysis_of_Many_Rotor_Systems/quant_rotor/CCC_integration_methods/Dense/thermofield_boltz_funcs.py:150: ComplexWarning: Casting complex values to real discards the imaginary part
  v = np.asarray(v, dtype=float)


In [7]:
tau_arr = np.arange(0, 40.5, 0.5)
energy_arr = np.zeros_like(tau_arr)

In [8]:
W = eig_vec_TF

th =  np.zeros((81))
th[0] = 1

In [9]:
t = 3

In [10]:
tau_0 = t

exp_term = np.zeros((81,81))

for i in range(81):
    exp_term[i, i] = np.exp(-eig_val_TF[i] * tau_0)

gs = (W @ exp_term) @ (W.T @ th)
ev = np.zeros((81, 2))
ev[:, 0] = gs
ev[:, 1] = gs

t_1_exact = t_1_amplitude_guess_ground_state(state_TF, site, ev, [0, 1])
t_2_exact = t_2_amplitude_guess_ground_state(state_TF, site, ev, [0, 1])

/var/folders/vt/f4sbj2d11gbbb9q5816khz2m0000gn/T/ipykernel_6316/3660228695.py:10: ComplexWarning: Casting complex values to real discards the imaginary part
  ev[:, 0] = gs
/var/folders/vt/f4sbj2d11gbbb9q5816khz2m0000gn/T/ipykernel_6316/3660228695.py:11: ComplexWarning: Casting complex values to real discards the imaginary part
  ev[:, 1] = gs


In [11]:
K_e_copy = np.copy(K_tilda)
V_e_copy_xy = np.copy(V_tilda)
V_e_copy_yx = np.copy(V_tilda)

In [12]:
K_e_comb = K_e_copy
V_e_comb_xy = V_e_copy_xy
V_e_comb_yx = V_e_copy_yx

In [13]:
t_1_new = np.copy(t_1_exact)
t_2_new = np.copy(t_2_exact)

In [14]:
dt = 1e-5
tau_0 = t - dt

exp_term = np.zeros((81,81))

for i in range(81):
    exp_term[i, i] = np.exp(-eig_val_TF[i] * tau_0)

gs = (W @ exp_term) @ (W.T @ th)
ev = np.zeros((81, 2))
ev[:, 0] = gs
ev[:, 1] = gs

t_1_0 = t_1_amplitude_guess_ground_state(state_TF, site, ev, [0, 1])
t_2_0 = t_2_amplitude_guess_ground_state(state_TF, site, ev, [0, 1])

# E = amplitute_energy(site, state_2, periodic, K_tilda, V_tilda.reshape(state_2, state_2, state_2, state_2) ,V_tilda.reshape(state_2, state_2, state_2, state_2), t_1.reshape(site, a, 1), t_2.reshape(site, site, a, a, 1, 1))
tau_1 = t + 0.00001

exp_term = np.zeros((81,81))

for i in range(81):
    exp_term[i, i] = np.exp(-eig_val_TF[i] * tau_1)

gs = (W @ exp_term) @ (W.T @ th)
ev = np.zeros((81, 2))
ev[:, 0] = gs
ev[:, 1] = gs

t_1_1 = t_1_amplitude_guess_ground_state(state_TF, site, ev, [0, 1])
t_2_1 = t_2_amplitude_guess_ground_state(state_TF, site, ev, [0, 1])

# E = amplitute_energy(site, state_2, periodic, K_tilda, V_tilda.reshape(state_2, state_2, state_2, state_2) ,V_tilda.reshape(state_2, state_2, state_2, state_2), t_1.reshape(site, a, 1), t_2.reshape(site, site, a, a, 1, 1))
dt_1_dt =  (t_1_1 - t_1_0)/(2*dt)
dt_2_dt =  (t_2_1 - t_2_0)/(2*dt)

/var/folders/vt/f4sbj2d11gbbb9q5816khz2m0000gn/T/ipykernel_6316/807302270.py:11: ComplexWarning: Casting complex values to real discards the imaginary part
  ev[:, 0] = gs
/var/folders/vt/f4sbj2d11gbbb9q5816khz2m0000gn/T/ipykernel_6316/807302270.py:12: ComplexWarning: Casting complex values to real discards the imaginary part
  ev[:, 1] = gs
/var/folders/vt/f4sbj2d11gbbb9q5816khz2m0000gn/T/ipykernel_6316/807302270.py:27: ComplexWarning: Casting complex values to real discards the imaginary part
  ev[:, 0] = gs
/var/folders/vt/f4sbj2d11gbbb9q5816khz2m0000gn/T/ipykernel_6316/807302270.py:28: ComplexWarning: Casting complex values to real discards the imaginary part
  ev[:, 1] = gs


## CI Version 1

In [15]:
p = state_TF
i = 1
a = p - i

epsilon = np.diag(K_e_comb)

In [16]:
params = SimulationParams(
    a=a,
    i=i,
    p=p,  # These can be the same as `a + i` or chosen independently
    site=site,
    state=state_TF,
    i_method=3,
    gap=False,
    gap_site=3,
    epsilon=epsilon,
    periodic=periodic,
)

tensors = TensorData(
    t_a_i_tensor=t_1_new,
    t_ab_ij_tensor=t_2_new,
    h_full=K_e_comb,
    v_full_xy=V_e_comb_xy.reshape(state_TF, state_TF, state_TF, state_TF),
    v_full_yx=V_e_comb_yx.reshape(state_TF, state_TF, state_TF, state_TF),
)

qs = QuantumSimulation(params, tensors)

In [17]:
V.reshape(state, state, state, state)[0, 0]

array([[ 0.   +0.j,  0.   +0.j,  0.   +0.j],
       [ 0.   +0.j, -0.075+0.j, -0.025+0.j],
       [ 0.   +0.j, -0.025+0.j, -0.075+0.j]])

In [18]:
V_e_comb_xy.reshape(state_TF, state_TF, state_TF, state_TF)[0, 0].real

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.025     , -0.00833333, -0.00833333,  0.        , -0.025     ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.00833333, -0.025     , -0.025     ,  0.        , -0.00833333,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.00833333, -0.025     , -0.025     ,  0.        , -0.00833333,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.025     , -0.00833333, -0.00833333,  0.        , -0.025     ,  0.   

In [19]:
energy_1 = 0
energy_2 = 0

for site_x in range(site):
    energy_1 += np.einsum("ip, pi->", qs.h_term(i, p), qs.B_term(i, site_x))

    for site_y in range(site):
        if site_x < site_y:
            # if abs(site_x - site_y) == 1:
            # noinspection SpellCheckingInspection
            energy_2 += np.einsum(
                "ijab, abij->",
                qs.v_term(i, i, a, a, site_x, site_y),
                qs.t_term_2(site_x, site_y),
            )
            # noinspection SpellCheckingInspection
            energy_2 += np.einsum(
                "ijpq, pi, qj->",
                qs.v_term(i, i, p, p, site_x, site_y),
                qs.B_term(i, site_x),
                qs.B_term(i, site_y),
            )

In [20]:
C_1 = np.einsum(
    "pi, qj->pqij",
    qs.B_term(i, 0),
    qs.B_term(i, 1))

In [21]:
C_1[1:, 1:] += t_2_new[0, 1]

In [22]:
L_h = np.einsum(
    "pP, Pqij->pqij",
    qs.h_term(p, p),
    C_1)

L_h += np.einsum(
    "pQij, qQ->pqij",
    C_1,
    qs.h_term(p, p))

L_h -= energy_1 * C_1

In [23]:
L_V = np.einsum("pqPQ, PQij -> pqij", qs.v_term(p, p, p, p, 0, 1), C_1) - energy_2 * C_1

In [24]:
L_h.reshape(9, 9)

array([[ 0.        +0.j, -0.01097025+0.j, -0.04939527+0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j,  0.00099137+0.j, -0.        +0.j,  0.00099137+0.j],
       [-0.01097025+0.j, -0.00198741+0.j, -0.01411339+0.j, -0.        +0.j, -0.        +0.j,  0.        +0.j,  0.00007582+0.j,  0.        +0.j,  0.00007455+0.j],
       [-0.04939527+0.j, -0.01411339+0.j, -0.06240069+0.j, -0.        +0.j, -0.        +0.j,  0.        +0.j,  0.00033838+0.j,  0.        +0.j,  0.00033866+0.j],
       [-0.        +0.j,  0.        +0.j,  0.        +0.j,  0.01000115+0.j,  0.00334272+0.j,  0.00903946+0.j,  0.        +0.j,  0.02698557+0.j,  0.        +0.j],
       [-0.        +0.j, -0.        +0.j,  0.        +0.j,  0.00334272+0.j,  0.01000115+0.j,  0.02698557+0.j,  0.        +0.j,  0.00903946+0.j,  0.        +0.j],
       [ 0.        +0.j,  0.        +0.j, -0.        +0.j,  0.00903946+0.j,  0.02698557+0.j,  0.08963926+0.j,  0.        +0.j,  0.02987975+0.j,  0.        +0.j],
       [ 0.00099137+0.j,  0.

In [25]:
L_V.reshape(9, 9)

array([[ 0.        +0.j,  0.00007411+0.j,  0.00033369+0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j, -0.00094716+0.j, -0.        +0.j, -0.00094716+0.j],
       [ 0.00007411+0.j, -0.00104464+0.j,  0.00014362+0.j, -0.        +0.j,  0.        +0.j,  0.        +0.j, -0.00007667+0.j,  0.        +0.j, -0.00007516+0.j],
       [ 0.00033369+0.j,  0.00014362+0.j, -0.00042985+0.j, -0.        +0.j, -0.        +0.j, -0.        +0.j, -0.00034165+0.j, -0.        +0.j, -0.00034199+0.j],
       [-0.        +0.j, -0.        +0.j,  0.        +0.j, -0.00719334+0.j, -0.00241179+0.j, -0.00863842+0.j, -0.        +0.j, -0.02577589+0.j, -0.        +0.j],
       [-0.        +0.j, -0.        +0.j,  0.        +0.j, -0.00241179+0.j, -0.00719334+0.j, -0.02577589+0.j,  0.        +0.j, -0.00863842+0.j, -0.        +0.j],
       [-0.        +0.j, -0.        +0.j,  0.        +0.j, -0.00863842+0.j, -0.02577589+0.j, -0.09268908+0.j,  0.        +0.j, -0.03089636+0.j, -0.        +0.j],
       [-0.00094716+0.j, -0.

## CI Version 2

In [26]:
p = state_TF
i = 1
a = p - i

epsilon = np.diag(K_e_comb)
params_linked = SimulationParams_linked(
    a=a,
    i=i,
    p=p,  # These can be the same as `a + i` or chosen independently
    site=site,
    state=state_TF,
    i_method=3,
    gap=False,
    gap_site=3,
    epsilon=epsilon,
    periodic=periodic,
)

tensors_linked = TensorData_linked(
    t_a_i_tensor=t_1_new,
    t_ab_ij_tensor=t_2_new,
    h_full=K_e_comb,
    v_full_xy=V_e_comb_xy.reshape(state_TF, state_TF, state_TF, state_TF),
    v_full_yx=V_e_comb_yx.reshape(state_TF, state_TF, state_TF, state_TF),
)

qs_linked = QuantumSimulation_linked(params_linked, tensors_linked)

In [27]:
def R_1_CI_func(x: int):

    y = (x + 1) % 2

    r_1 = np.einsum("ap, pi -> ai", qs_linked.h_term(a, p), qs_linked.B_term(i, x))

    r_1 += np.einsum("jb, bj, ai -> ai", qs_linked.h_term(i,a), qs_linked.t_term_1(y), qs_linked.t_term_1(x))
    r_1 += np.einsum("ajcd, cdij-> ai", qs_linked.v_term(a, i, a, a, x, y), qs_linked.t_term_2(x, y))
    r_1 += np.einsum("ajpq, pi, qj -> ai", qs_linked.v_term(a, i, p, p, x, y), qs_linked.B_term(i, x), qs_linked.B_term(i, y))
    r_1 += np.einsum("jb, abij -> ai", qs_linked.h_term(i, a), qs_linked.t_term_2(x, y))
    # r_1 += np.einsum("jb, abij -> ai", qs_linked.h_term(i, a), qs_linked.t_term_2(y, x))
    r_1 -= (qs_linked.ec1(x) + qs_linked.ec1(y) + qs_linked.ec2(x, y)) * tensors_linked.t_a_i_tensor[x]

    return r_1

# def R_1_CI_func(x: int):

#     y = (x + 1) % 2

#     r_1 = np.einsum("ap, pi -> ai", qs_linked.h_term(a, p), qs_linked.B_term(i, x))

#     r_1 += np.einsum("ajcd, cdij-> ai", qs_linked.v_term(a, i, a, a, x, y), qs_linked.t_term_2(x, y))
#     r_1 += np.einsum("ajpq, pi, qj -> ai", qs_linked.v_term(a, i, p, p, x, y), qs_linked.B_term(i, x), qs_linked.B_term(i, y))

#     r_1 -= (qs_linked.ec1(x) + qs_linked.ec1(y) + qs_linked.ec2(x, y)) * tensors_linked.t_a_i_tensor[x]

#     return r_1

def R_2_CI_func(x: int, y: int):

    r_2 = np.einsum("ap, pi, bj -> abij", qs_linked.h_term(a, p), qs_linked.B_term(i, x), qs_linked.t_term_1(y))
    r_2 += np.einsum("ac, cbij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y))
    r_2 += np.einsum("bq, ai, qj -> abij", qs_linked.h_term(a, p), qs_linked.t_term_1(x), qs_linked.B_term(i, y))
    r_2 += np.einsum("bd, adij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y))

    r_2 += np.einsum("abpq, pi, qj -> abij", qs_linked.v_term(a, a, p, p, x, y), qs_linked.B_term(i, x), qs_linked.B_term(i, y))
    r_2 += np.einsum("abcd, cdij -> abij", qs_linked.v_term(a, a, a, a, x, y), qs_linked.t_term_2(x, y))

    r_2 -= (np.einsum("ai, bj -> abij", qs_linked.t_term_1(x), qs_linked.t_term_1(y)) + tensors_linked.t_ab_ij_tensor[x, y])*(qs_linked.ec1(x) + qs_linked.ec1(y) + qs_linked.ec2(x, y))

    return r_2

In [28]:
x = 0
y = 1

In [29]:
term_2 = np.einsum("bd, adij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y)).reshape(a, a)

In [30]:
term_1 = np.einsum("ac, cbij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(x, y)).reshape(a, a)

In [31]:
R_10 = np.einsum("ap, pc, cbij->abij",qs.A_term(a, 1),qs.h_term(p, a),qs.t_term_2(1, 0)).reshape(a, a)
R_01 = np.einsum("ap, pc, cbij->abij",qs.A_term(a, 0),qs.h_term(p, a),qs.t_term_2(0, 1)).reshape(a, a)

In [32]:
R_1_CI = np.zeros((site, a, i), dtype=complex)
R_2_CI = np.zeros((site, site, a, a, i, i), dtype=complex)

In [33]:
for x in range(site):
    R_1_CI[x] =  R_1_CI_func(x)
    for y in range(site):
        if x < y:
            R_2_CI[x, y] = R_2_CI_func(x, y)
            R_2_CI[y, x] = R_2_CI[x, y].reshape(a, a).T.reshape(a, a, i, i)

## CCC

In [34]:
T_1_f, T_2_f, R_1_f, R_2_f = t_periodic_fast(site, state_TF, K_e_comb, V_e_comb_xy.reshape(state_TF, state_TF, state_TF, state_TF), V_e_comb_yx.reshape(state_TF, state_TF, state_TF, state_TF), Import_t=True, t_1_import=np.copy(t_1_new.reshape(site, a)), t_2_import=np.copy(t_2_new.reshape(site, site, a, a)), periodic=periodic, one_cicle=True)

In [35]:
T_1, T_2, R_1, R_2 = t_periodic(site, state_TF, K_e_comb, V_e_comb_xy.reshape(state_TF, state_TF, state_TF, state_TF), V_e_comb_yx.reshape(state_TF, state_TF, state_TF, state_TF), "original", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

original


In [36]:
T_1_linked, T_2_linked, R_1_linked, R_2_linked = t_periodic(site, state_TF, K_e_comb, V_e_comb_xy.reshape(state_TF, state_TF, state_TF, state_TF), V_e_comb_yx.reshape(state_TF, state_TF, state_TF, state_TF), "linked", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

linked


In [37]:
T_1_transformed, T_2_transformed, R_1_transformed, R_2_transformed = t_periodic(site, state_TF, K_e_comb, V_e_comb_xy.reshape(state_TF, state_TF, state_TF, state_TF), V_e_comb_yx.reshape(state_TF, state_TF, state_TF, state_TF), "transformed", Import_t=True, t_1_import=np.copy(t_1_new), t_2_import=np.copy(t_2_new), periodic=periodic, one_cicle=True)

transformed


In [38]:
R_E = L_h + L_V
R_E_x = R_E[1:, 0, 0, 0].reshape(a, i)
R_E_y = R_E[0, 1:, 0, 0].reshape(a, i)
R_E_xy = R_E[1:, 1:, 0, 0].reshape(a, a, i, i)

In [39]:
np.max(np.abs(R_E))

np.float64(0.06283053880476251)

In [40]:
R_E_xy_cor = R_E_xy - np.einsum("ai, bj -> abij", R_E_x, t_1_new[1]) - np.einsum("ai, bj -> abij", R_E_y, t_1_new[0])

## Output

In [41]:
A_h = np.einsum("ap, pc ->ac", qs.A_term(a, 1), qs.h_term(p, a))

In [42]:
R = np.einsum("ap, pc, cbij->abij",qs.A_term(a, 0),qs.h_term(p, a),qs.t_term_2(0, 1),)

In [43]:
r_2 = np.einsum("ac, cbij -> abij", qs_linked.h_term(a, a), qs_linked.t_term_2(0, 1))

In [44]:
print("CI site x:", np.max(np.abs(R_E_x - R_1_CI[0])))
print("CI site y:", np.max(np.abs(R_E_y - R_1_CI[1])))
print("CI site xy:", np.max(np.abs(R_E_xy - R_2_CI[0, 1])))

CI site x: 8.326672684688674e-17
CI site y: 2.7755575615628914e-17
CI site xy: 2.0816681711721685e-17


In [55]:
# print("CI1/ FF  R1:", np.max(np.abs(dt_1_dt[0] + R_1_CI[0])))
# print("CI2/ FF R1:", np.max(np.abs(dt_1_dt[0] + R_E_x)))
# print("CCC/ FF R1:", np.max(np.abs(dt_1_dt[0] + R_1[0])))
print("CCC/ FF R1:", np.max(np.abs(dt_1_dt[0] + R_1_f[0].reshape(a, 1))))
print("CCC/ FF R1:", np.max(np.abs(R_1[0] - R_1_f[0].reshape(a, 1))))
# print("CCC/ FF R1:", np.max(np.abs(dt_1_dt[0] + R_1_linked[0])))
# print("CCC/ FF R1:", np.max(np.abs(dt_1_dt[0] + R_1_transformed[0])))

print("\n")

# print("CI1/ FF R2:", np.max(np.abs(dt_2_dt[0, 1] + R_2_CI[0, 1])))
# print("CI2/ FF R2:", np.max(np.abs(dt_2_dt[0, 1] + R_E_xy_cor)))
# print("CCC/ FF R2:", np.max(np.abs(dt_2_dt[0, 1] + R_2[0, 1])))
print("CCC/ FF R2:", np.max(np.abs(dt_2_dt[0, 1] + R_2_f[0, 1].reshape(a, a, 1, 1))))
print("CCC/ FF R2:", np.max(np.abs(R_2[0, 1] - R_2_f[0, 1].reshape(a, a, 1, 1))))
# print("CCC/ FF R2:", np.max(np.abs(dt_2_dt[0, 1] + R_2_linked[0, 1])))
# print("CCC/ FF R2:", np.max(np.abs(dt_2_dt[0, 1] + R_2_transformed[0, 1])))

CCC/ FF R1: 7.000892920938639e-13
CCC/ FF R1: 2.7755575615628914e-17


CCC/ FF R2: 2.8781676952231543e-12
CCC/ FF R2: 1.3877787807814457e-17


In [46]:
R_1[0]

array([[-0.01089614+0.j],
       [-0.04906158+0.j],
       [-0.        +0.j],
       [-0.        +0.j],
       [ 0.        +0.j],
       [ 0.00004421+0.j],
       [ 0.        +0.j],
       [ 0.00004421+0.j]])

In [47]:
R_1_f[0].reshape(a, i)

array([[-0.01089614+0.j],
       [-0.04906158+0.j],
       [-0.        +0.j],
       [-0.        +0.j],
       [ 0.        +0.j],
       [ 0.00004421+0.j],
       [ 0.        +0.j],
       [ 0.00004421+0.j]])

In [48]:
R_2_transformed[0, 1].shape

(8, 8, 1, 1)

In [49]:
np.max(np.abs(R_1))
np.max(np.abs(R_2))

np.float64(0.0030498134674811564)

In [50]:
# print("Connected/Linked R_1:" ,np.max(np.abs(R_1_linked - R_1)))
# print("Connected/Linked R_2:" ,np.max(np.abs(R_2_linked - R_2)), "\n")

# print("Connected/Transformed R_1:" ,np.max(np.abs(R_1_transformed - R_1)))
# print("Connected/Transformed R_2:" ,np.max(np.abs(R_2_transformed - R_2)), "\n")

# print("Linked/Transformed R_1:" ,np.max(np.abs(R_1_transformed - R_1_linked)))
# print("Linked/Transformed R_2:" ,np.max(np.abs(R_2_transformed - R_2_linked)))

In [51]:
print("R_E site x:" ,np.max(np.abs(R_1[0] - R_1_CI[0])))
print("R_E site y:" ,np.max(np.abs(R_1[0] - R_1_CI[0])))
print("R_E site xy:" ,np.max(np.abs(R_2[0, 1].reshape(a, a) - R_2_CI[0, 1].reshape(a, a))))
# print("R_E site xy corrected:" ,np.max(np.abs(R_2[0, 1].reshape(a, a) - R_2_CI[0, 1].reshape(a, a))))

R_E site x: 4.85722573273506e-17
R_E site y: 4.85722573273506e-17
R_E site xy: 0.06278473543543453


In [52]:
0.16227420219160713

0.16227420219160713

In [53]:
0.06156605498959347

0.06156605498959347

In [54]:
0.15405505531135882

0.15405505531135882